<a href="https://colab.research.google.com/github/alyairawan02/publicdiscussion_analysis/blob/main/Mengetahui_Topik_Sentimen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# 1. Baca ketiga file
df1 = pd.read_excel('data_final_diproses.xlsx')
df2 = pd.read_excel('df_final_with_encoded_label.xlsx')

# 2. Gabungkan ketiga dataframe
df = pd.concat([df1, df2], ignore_index=True)

# 3. Lihat struktur data hasil gabungan
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4685 entries, 0 to 4684
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   created_at        4685 non-null   object 
 1   full_text         4685 non-null   object 
 2   cleaning          4685 non-null   object 
 3   case_folding      4685 non-null   object 
 4   normalisasi       4685 non-null   object 
 5   tokenize          4685 non-null   object 
 6   stopword removal  4685 non-null   object 
 7   stemming_data     4685 non-null   object 
 8   sent_label        4685 non-null   object 
 9   sent_score        4685 non-null   float64
 10  encoded_label     4685 non-null   int64  
 11  username          3304 non-null   object 
dtypes: float64(1), int64(1), object(10)
memory usage: 439.3+ KB


In [ ]:
label_map = {1: 'NEUTRAL', 0: 'NEGATIVE', 2: 'POSITIVE'}

for kode, label_nama in label_map.items():
    subset = df[df['encoded_label'] == kode]
    subset.to_csv(f'data_{label_nama}.csv', index=False)
    print(f"File untuk label '{label_nama}' disimpan sebagai data_{label_nama}.csv")

File untuk label 'NEUTRAL' disimpan sebagai data_NEUTRAL.csv
File untuk label 'NEGATIVE' disimpan sebagai data_NEGATIVE.csv
File untuk label 'POSITIVE' disimpan sebagai data_POSITIVE.csv


In [ ]:
!pip install Sastrawi

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

df = pd.read_csv('data_NEGATIVE.csv')

# Stopwords bahasa Indonesia dari Sastrawi
stop_factory = StopWordRemoverFactory()
stopwords = stop_factory.get_stop_words()  # Ini sudah list, bukan set

tambahan_stopwords = [
    'yg', 'aja', 'sehat', 'sehatkan', 'ga', 'tidak', 'baik',
    'bagus', 'mantap', 'top', 'enak', 'co', 'https', 'http',
]
stopwords.extend(tambahan_stopwords)

# Normalisasi teks
df['cleaning'] = df['cleaning'].astype(str).str.lower()

# TF-IDF
tfidf = TfidfVectorizer(max_features=1000, stop_words=stopwords)
tfidf_matrix = tfidf.fit_transform(df['cleaning'])

# Hitung rata-rata skor TF-IDF
mean_tfidf = tfidf_matrix.mean(axis=0).A1
words = tfidf.get_feature_names_out()

# Ambil top N kata
top_n = 5
top_indices = mean_tfidf.argsort()[-top_n:][::-1]
top_words = [(words[i], mean_tfidf[i]) for i in top_indices]

# Tampilkan hasil
print(f"\nTop {top_n} kata dari sentimen negatif:")
for word, score in top_words:
    print(f"{word} (TF-IDF: {score:.4f})")


Top 5 kata dari sentimen negatif:
mbg (TF-IDF: 0.0949)
anak (TF-IDF: 0.0383)
program (TF-IDF: 0.0309)
bangsa (TF-IDF: 0.0236)
keracunan (TF-IDF: 0.0210)


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

df = pd.read_csv('data_POSITIVE.csv')

# Stopwords bahasa Indonesia dari Sastrawi
stop_factory = StopWordRemoverFactory()
stopwords = stop_factory.get_stop_words()  # Ini sudah list, bukan set

tambahan_stopwords = [
    'yg', 'aja', 'ga', 'tidak', 'baik', 'co', 'https', 'http', 'bergizi',
    'gratis', 'papua', 'papuasehat', 'makan'
]
stopwords.extend(tambahan_stopwords)

# Normalisasi teks
df['cleaning'] = df['cleaning'].astype(str).str.lower()

# TF-IDF
tfidf = TfidfVectorizer(max_features=1000, stop_words=stopwords)
tfidf_matrix = tfidf.fit_transform(df['cleaning'])

# Hitung rata-rata skor TF-IDF
mean_tfidf = tfidf_matrix.mean(axis=0).A1
words = tfidf.get_feature_names_out()

# Ambil top N kata
top_n = 5
top_indices = mean_tfidf.argsort()[-top_n:][::-1]
top_words = [(words[i], mean_tfidf[i]) for i in top_indices]

# Tampilkan hasil
print(f"\nTop {top_n} kata dari sentimen positif:")
for word, score in top_words:
    print(f"{word} (TF-IDF: {score:.4f})")


Top 5 kata dari sentimen positif:
mbg (TF-IDF: 0.0826)
program (TF-IDF: 0.0739)
dukung (TF-IDF: 0.0540)
anakanak (TF-IDF: 0.0291)
sehat (TF-IDF: 0.0278)


In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

df1 = pd.read_excel('data_final_diproses.xlsx')
df2 = pd.read_excel('df_final_with_encoded_label.xlsx')

df = pd.concat([df1, df2], ignore_index=True)

label_map = {1: 'NEUTRAL', 0: 'NEGATIVE', 2: 'POSITIVE'}

model = SentenceTransformer("all-MiniLM-L6-v2")

# Proses untuk setiap label
for kode, label_nama in label_map.items():
    subset = df[df['encoded_label'] == kode]
    sentences = subset['cleaning'].dropna().tolist()

    print(f"\nLabel: {label_nama} | Total kalimat: {len(sentences)}")

    if len(sentences) < 2:
        print(" - Kalimat terlalu sedikit untuk dihitung similarity.")
        continue

    embeddings = model.encode(sentences)

    similarity_matrix = cosine_similarity(embeddings)

    threshold = 0.8
    for i in range(len(sentences)):
        for j in range(i+1, len(sentences)):
            score = similarity_matrix[i][j]
            if score >= threshold:
                print(f" - Pair: ({i}, {j}) | Similarity: {score:.4f}")
                print(f"    Sentence 1: {sentences[i]}")
                print(f"    Sentence 2: {sentences[j]}\n")


Label: NEUTRAL | Total kalimat: 1142


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Streaming output truncated to the last 5000 lines.
    Sentence 2: Warga pedalaman Papua terbantu dengan layanan kesehatan gratis dari Yonif  Kostrad PapuaSehat 

 - Pair: (1053, 1125) | Similarity: 0.8378
    Sentence 1: Sekolah Dibakar Semangat Belajar Tetap Menyala Pasca aksi pembakaran SD oleh OPM di Intan Jaya MBGPapua PapuaBangkit SekolahDarurat TNIuntukPapua PendidikanPapua MakanBergiziGratis StopKekerasanPapua GenerasiEmasIndonesia GiziAnak PapuaSehat 
    Sentence 2: Satgas Yonif  Kostrad sediakan pengobatan gratis bagi masyarakat pedalaman Papua PapuaSehat 

 - Pair: (1053, 1127) | Similarity: 0.8334
    Sentence 1: Sekolah Dibakar Semangat Belajar Tetap Menyala Pasca aksi pembakaran SD oleh OPM di Intan Jaya MBGPapua PapuaBangkit SekolahDarurat TNIuntukPapua PendidikanPapua MakanBergiziGratis StopKekerasanPapua GenerasiEmasIndonesia GiziAnak PapuaSehat 
    Sentence 2: Pendidikan dan Kesehatan Untuk Masa Depan Papua PapuaCerdas PapuaSehat 

 - Pair: (1053, 1128) | Similarity

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Streaming output truncated to the last 5000 lines.
    Sentence 2: yaaa ya mau ini percaya IKN saat nanti butuh Pertamina Agak Dimana apalagi dana MBG sokongan korupsi ragu pada adalagi

 - Pair: (619, 970) | Similarity: 0.9839
    Sentence 1:  Sudah ku duga MBG bakal jdi celah untuk korupsi
    Sentence 2: ku jdi Sudah untuk MBG bakal korupsi duga celah

 - Pair: (620, 1131) | Similarity: 0.9807
    Sentence 1:  CSR Lawless gt
    Sentence 2: Lawless CSR gt

 - Pair: (620, 1356) | Similarity: 0.9790
    Sentence 1:  CSR Lawless gt
    Sentence 2: gt Lawless CSR

 - Pair: (622, 1132) | Similarity: 0.9887
    Sentence 1:   Baiknya gini sih sekalian bantu kantin sekolah juga Semenjak MBG pada sepitutup
    Sentence 2: gini juga sekalian bantu Baiknya sih kantin pada sekolah sepitutup Semenjak MBG

 - Pair: (622, 1430) | Similarity: 0.9884
    Sentence 1:   Baiknya gini sih sekalian bantu kantin sekolah juga Semenjak MBG pada sepitutup
    Sentence 2: kantin Semenjak sekalian sepitutup ju

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Streaming output truncated to the last 5000 lines.
 - Pair: (1565, 1793) | Similarity: 0.8547
    Sentence 1: yang depan cerdas memastikan kuat masa mereka bersama Papua citacita kita adalah anakanak lebih MakanBergiziGratis fondasi dan sehat untuk menjadi penting Mewujudkan cerah Makan MBGPapua PapuaSehat Program tumbuh bagi Gratis Bergizi
    Sentence 2: Program Papua anakanak dengan Papua untuk MakanBergiziGratis makanan Program Makan meningkatkan Gratis nyata PapuaSehat sangat MakanBergizi untuk yang GenerasiSehat bergizi akses dibutuhkan memberikan StuntingFree Bergizi langkah Mendukung kesejahteraan Masyarakat menjadi

 - Pair: (1565, 1803) | Similarity: 0.9065
    Sentence 1: yang depan cerdas memastikan kuat masa mereka bersama Papua citacita kita adalah anakanak lebih MakanBergiziGratis fondasi dan sehat untuk menjadi penting Mewujudkan cerah Makan MBGPapua PapuaSehat Program tumbuh bagi Gratis Bergizi
    Sentence 2: Makan Bergizi Gratis langkah untuk yang cerdas makanan seha

In [ ]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 3.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

label = 'NEGATIVE'

# --- Setup stopwords ---
factory = StopWordRemoverFactory()
stopwords = factory.get_stop_words()

# Tambahkan custom stopwords
tambahan_stopwords = [
    'yg', 'aja', 'sehat', 'sehatkan', 'ga', 'tidak', 'baik',
    'bagus', 'mantap', 'top', 'enak', 'co', 'https', 'http',
    'mbg', 'program', 'bangsa', 'bergizi', 'sekolah', 'makan', 'gratis'
]
stopwords.extend(tambahan_stopwords)


def extract_top_bigrams(texts, top_n=10):
    vectorizer = CountVectorizer(ngram_range=(2, 2), stop_words=stopwords)
    X = vectorizer.fit_transform(texts)
    bigram_counts = X.sum(axis=0).A1
    bigrams = vectorizer.get_feature_names_out()
    bigram_freq = list(zip(bigrams, bigram_counts))
    bigram_freq.sort(key=lambda x: x[1], reverse=True)
    return bigram_freq[:top_n]

filename = f'data_{label}.csv'

try:
    df = pd.read_csv(filename)
    texts = df['cleaning'].astype(str).tolist()

    top_bigrams = extract_top_bigrams(texts, top_n=10)
    print(f"\nTop 5 Bigram untuk Sentimen: {label}")
    for bigram, freq in top_bigrams:
        if freq >= 8:
            print(f" - {bigram} (frekuensi: {freq})")

except FileNotFoundError:
    print(f"File {filename} tidak ditemukan.")


Top 5 Bigram untuk Sentimen: NEGATIVE
 - prioritas utama (frekuensi: 14)
 - prioritas pendukung (frekuensi: 11)
 - berbasis rakyat (frekuensi: 9)
 - pendidikan kesehatan (frekuensi: 9)
 - gue gue (frekuensi: 8)
 - makanan basi (frekuensi: 8)
 - otak terbang (frekuensi: 8)
 - siswa keracunan (frekuensi: 8)
 - siswa smp (frekuensi: 8)


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

label = 'POSITIVE'

factory = StopWordRemoverFactory()
stopwords = factory.get_stop_words()

tambahan_stopwords = [
    'yg', 'aja', 'ga', 'tidak', 'baik', 'co', 'https', 'http', 'bergizi',
    'gratis', 'papua', 'papuasehat', 'makan'
]
stopwords.extend(tambahan_stopwords)

def extract_all_bigrams(texts):
    vectorizer = CountVectorizer(ngram_range=(2, 2), stop_words=stopwords)
    X = vectorizer.fit_transform(texts)
    bigram_counts = X.sum(axis=0).A1
    bigrams = vectorizer.get_feature_names_out()
    bigram_freq = list(zip(bigrams, bigram_counts))
    bigram_freq.sort(key=lambda x: x[1], reverse=True)
    return bigram_freq

filename = f'data_{label}.csv'

try:
    df = pd.read_csv(filename)
    texts = df['cleaning'].astype(str).tolist()

    all_bigrams = extract_all_bigrams(texts)
    print(f"\nBigram dengan frekuensi 18 untuk Sentimen: {label}")
    ada = False
    for bigram, freq in all_bigrams:
        if freq >=8:
            print(f" - {bigram} (frekuensi: {freq})")
            ada = True
    if not ada:
        print("Tidak ada bigram dengan frekuensi 18 ditemukan.")

except FileNotFoundError:
    print(f"File {filename} tidak ditemukan.")


Bigram dengan frekuensi 18 untuk Sentimen: POSITIVE
 - program mbg (frekuensi: 194)
 - masa depan (frekuensi: 90)
 - dukung program (frekuensi: 61)
 - tumbuh sehat (frekuensi: 42)
 - presiden prabowo (frekuensi: 39)
 - mbg program (frekuensi: 37)
 - sehat cerdas (frekuensi: 36)
 - lanjutkan mbg (frekuensi: 34)
 - pelatihan sppi (frekuensi: 26)
 - mbg bikin (frekuensi: 25)
 - bersama program (frekuensi: 22)
 - program dukung (frekuensi: 22)
 - depan lebih (frekuensi: 21)
 - anak sehat (frekuensi: 18)
 - dukung bersama (frekuensi: 18)
 - mbg sangat (frekuensi: 18)
 - asupan gizi (frekuensi: 17)
 - gizi cukup (frekuensi: 17)
 - mbg jadi (frekuensi: 17)
 - makanan mbg (frekuensi: 16)
 - mbg presiden (frekuensi: 16)
 - sangat bermanfaat (frekuensi: 16)
 - anakanak tumbuh (frekuensi: 15)
 - bersama dukung (frekuensi: 15)
 - jadi mbg (frekuensi: 15)
 - mbg lebih (frekuensi: 15)
 - program makanbergizigratis (frekuensi: 15)
 - program presiden (frekuensi: 15)
 - sehat kuat (frekuensi: 15)
 - 